# Clase 4 — Churn en GCP: de los datos al scoring batch

Esta notebook es el lab de la clase 4. La corremos en el **editor de Cloud Shell**, con el repo del
curso clonado. Cada paso es **un concepto, una celda que apretás, y un lugar de la consola donde ver
el recurso que apareció**.

**Antes de arrancar necesitás:**
- Tu cuenta de GCP activa y un proyecto con billing (eso lo hicimos en la Parte 0).
- Las APIs de Vertex y Cloud Storage ya prendidas (también Parte 0).
- Cloud Shell abierto, este repo clonado, y estar parado en la carpeta `notebooks/`.

Caso guía: **Customer Churn**. El usuario es el equipo de **CRM**. Queremos una lista de a quién
llamar primero, ordenada por `priority_score = churn_probability × MonthlyCharges`.

> Nota técnica: para hablar con la nube usamos las **librerías de Python de Google** (`google-cloud-storage`),
> que autentican solas con las credenciales de Cloud Shell. Es más confiable que llamar al comando
> `gcloud` desde el kernel del editor.

## Paso 0 — Dónde estás parado

Lo primero en la nube es siempre lo mismo: confirmar **en qué proyecto** vas a trabajar. Todo lo que
crees hoy (bucket, modelo) vive dentro de ese proyecto.

En el editor de Cloud Shell el proyecto **no siempre se autodetecta**. Si la celda imprime
`mlops-2026-itba`, cambiá esa línea por el ID que ves **arriba a la izquierda en la consola** (al lado
del logo de Google Cloud).

In [ ]:
import os
from google.cloud import storage

# En Cloud Shell a veces el proyecto viene en una variable de entorno; si no, ponelo a mano.
PROJECT_ID = (
    os.environ.get("GOOGLE_CLOUD_PROJECT")
    or os.environ.get("DEVSHELL_PROJECT_ID")
    or "mlops-2026-itba"          # <-- cambialo por el ID de TU proyecto
)

REGION = "us-central1"
BUCKET = f"{PROJECT_ID}-churn"

# El cliente autentica solo con las credenciales de Cloud Shell (ADC).
gcs = storage.Client(project=PROJECT_ID)

print("Proyecto:", PROJECT_ID)
print("Region:  ", REGION)
print("Bucket:  ", BUCKET)

## Paso 1 — La nube te responde

Un proyecto de GCP nace con casi todo **apagado**: cada servicio hay que **habilitar su API**. Las de
hoy (Vertex, Cloud Storage, Artifact Registry) las prendimos en la **Parte 0** desde la consola.

Para comprobar que todo está en orden, le pedimos a Cloud Storage la lista de buckets del proyecto.
Si esto responde, la API está prendida y estás autenticado.

> Si te faltara alguna API, se prende desde la consola (*APIs y servicios → Habilitar*) o en la
> terminal con `gcloud services enable aiplatform.googleapis.com storage.googleapis.com`.

In [ ]:
# Si esto lista buckets (aunque sea vacío), Cloud Storage responde y estas autenticado.
print("Buckets del proyecto:")
for b in gcs.list_buckets():
    print(" -", b.name)

## Paso 2 — El dato a la nube

El dato deja de vivir en una compu y pasa a **Cloud Storage**, el almacenamiento de objetos de GCP
(el equivalente al S3 de AWS). Primero verificamos que el CSV canónico esté sano (forma y checksum),
después creamos un **bucket** y subimos el archivo.

**Andá a ver:** en la consola, *Cloud Storage → Buckets → tu bucket → `raw/`*. Ahí está tu CSV.

In [ ]:
import sys, subprocess

# Verificamos forma y checksum del dataset canonico (esto corre local, no toca la nube).
check = subprocess.run([sys.executable, "../scripts/verify_dataset.py"],
                       capture_output=True, text=True)
print(check.stdout or check.stderr)

# Creamos el bucket si no existe.
bucket = gcs.bucket(BUCKET)
if bucket.exists():
    print("El bucket ya existe:", BUCKET)
else:
    bucket = gcs.create_bucket(BUCKET, location=REGION)
    print("Bucket creado:", BUCKET)

# Subimos el CSV.
bucket.blob("raw/Telco-Customer-Churn.csv").upload_from_filename(
    "../data/raw/Telco-Customer-Churn.csv"
)
print("Subido -> gs://%s/raw/Telco-Customer-Churn.csv" % BUCKET)

## Paso 3 — Entrenar

Acá pasás de **datos** a **modelo**. Reutilizamos el pipeline del caso guía (imputación, escalado,
one-hot y una regresión logística balanceada). Entrena en **segundos**: es el camino *a mano*, rápido
y reproducible. Vertex AutoML haría algo parecido, pero tardaría horas; lo miramos al final.

Cuando termine, leé el **AUC** y el **recall**: el recall es qué proporción de los que se van de
verdad estás agarrando. Para CRM, un recall alto significa **no dejar pasar** clientes en riesgo.

In [ ]:
sys.path.append("..")

import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score

from scripts.train_baseline import build_pipeline, load_training_data

DATA = Path("../data/raw/Telco-Customer-Churn.csv")
df = pd.read_csv(DATA)                      # crudo, lo usamos despues para customerID
X, y = load_training_data(DATA)             # features limpias + etiqueta 0/1

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline = build_pipeline()
pipeline.fit(X_tr, y_tr)

proba = pipeline.predict_proba(X_te)[:, 1]
pred = proba >= 0.7

{
    "roc_auc": round(roc_auc_score(y_te, proba), 3),
    "precision@0.7": round(precision_score(y_te, pred), 3),
    "recall@0.7": round(recall_score(y_te, pred), 3),
}

## Paso 4 — Registrar y versionar el modelo

Un modelo sin registrar es un archivo perdido. Le ponemos una **versión** con fecha y guardamos el
artefacto más sus métricas. Después lo subimos a Cloud Storage: así el modelo también **vive en la
nube**, con su linaje.

> Nota: el **Model Registry gestionado** de Vertex, para un modelo propio, pide además un contenedor
> de serving. Ese pozo lo dejamos para más adelante. Hoy el "registro" es el artefacto **versionado
> en Cloud Storage**, y el Registry real lo vas a ver en la demo de AutoML.

In [ ]:
import json
import joblib
from datetime import datetime, timezone

version = datetime.now(timezone.utc).strftime("churn-baseline-%Y%m%dT%H%M%SZ")
Path("../models").mkdir(parents=True, exist_ok=True)

joblib.dump(
    {"pipeline": pipeline, "metadata": {"model_version": version}},
    "../models/churn-baseline.joblib",
)
json.dump(
    {"model_version": version, "roc_auc": float(roc_auc_score(y_te, proba))},
    open("../models/churn-baseline-metrics.json", "w"),
    indent=2,
)
print("Modelo versionado:", version)

In [ ]:
# Subimos el modelo versionado y sus metricas a la nube.
bucket = gcs.bucket(BUCKET)
bucket.blob("models/churn-baseline.joblib").upload_from_filename("../models/churn-baseline.joblib")
bucket.blob("models/churn-baseline-metrics.json").upload_from_filename("../models/churn-baseline-metrics.json")

print("Modelo en gs://%s/models/" % BUCKET)
for b in gcs.list_blobs(BUCKET, prefix="models/"):
    print(" -", b.name)

## Paso 5 — Scoring batch

Esta es la inferencia **batch**: en vez de responder de a un cliente, scoreamos a **todos** de una
(acá usamos el conjunto de test como si fueran los clientes activos del mes) y armamos el ranking.

La prioridad no es solo la probabilidad de baja: es `churn_probability × MonthlyCharges`. Así CRM
llama primero a quien **se va y factura mucho**.

**Andá a ver:** después de subir el resultado, aparece en *Cloud Storage → `scored/`*.

In [ ]:
scored = X_te.copy()
scored["customerID"] = df.loc[X_te.index, "customerID"]
scored["churn_probability"] = proba
scored["priority_score"] = scored["churn_probability"] * scored["MonthlyCharges"]

ranking = scored.sort_values("priority_score", ascending=False)

# Guardamos el resultado del batch.
Path("../data/scored").mkdir(parents=True, exist_ok=True)
ranking.to_csv("../data/scored/batch-scored.csv", index=False)

ranking[["customerID", "churn_probability", "MonthlyCharges", "priority_score"]].head(10)

In [ ]:
# El resultado del batch tambien va a la nube.
gcs.bucket(BUCKET).blob("scored/batch-scored.csv").upload_from_filename(
    "../data/scored/batch-scored.csv"
)
print("Ranking en gs://%s/scored/" % BUCKET)
for b in gcs.list_blobs(BUCKET, prefix="scored/"):
    print(" -", b.name)

## Paso 6 — De score a decisión

El número no vale solo: vale porque **ordena una acción**. Aplicamos el umbral del caso: los clientes
con `churn_probability >= 0.7` entran a la **cola de retención**; al resto lo monitoreamos.

Esta es la lista concreta que CRM abre el lunes a la mañana.

In [ ]:
umbral = 0.7
cola = ranking[ranking["churn_probability"] >= umbral]

print(f"Clientes en cola de retencion (P >= {umbral}): {len(cola)} de {len(ranking)}")
cola[["customerID", "churn_probability", "MonthlyCharges", "priority_score"]].head(10)

## Paso 7 — Costos y limpieza

Regla de oro de la nube: **lo que prendés, cuesta**. Hoy tuvimos suerte: no desplegamos ningún
endpoint ni dejamos ninguna máquina prendida, así que no hay costo de serving corriendo. Lo único que
queda es el bucket con unos pocos MB, que entra holgado en el free tier.

Si querés dejar todo limpio, podés borrar el bucket de prueba. **Cuidado**: esto borra el dato, el
modelo y el resultado que subiste.

In [ ]:
# Descomenta solo si queres borrar TODO lo que subimos hoy:
# gcs.bucket(BUCKET).delete(force=True)
print("Nada quedo prendido cobrando. El bucket es lo unico que persiste.")

## Bonus — AutoML, para que lo veas (concepto)

Lo que hicimos a mano en segundos, Vertex lo puede hacer **solo**: le das el dataset tabular, le
decís que la columna a predecir es `Churn`, y prueba modelos por vos. El resultado es parecido; el
costo es **tiempo y crédito** (el entrenamiento tarda alrededor de dos horas), por eso no lo corremos
en vivo. En el ensayo, AutoML sacó **ROC AUC 0.895**, un poco mejor que nuestro `0.842` a mano.

El flujo gestionado, para que lo ubiques, es este (en clase lo vemos con capturas ya cocinadas):

```text
1) Crear un dataset tabular en Vertex desde el CSV en Cloud Storage.
2) Entrenar una clasificacion binaria con target = Churn.
3) Revisar las metricas de evaluacion que Vertex calcula solo.
4) Registrar el modelo en el Model Registry.
5) Correr un batch prediction sobre los clientes activos.
```

### ¿Querés reproducirlo vos?

No desde esta notebook: el AutoML se lanza **desde la TERMINAL** de Cloud Shell (ahí el SDK
`aiplatform` tiene proyecto y credenciales; el kernel del editor no). Con el dataset tabular ya creado
en la consola, se corre con dos scripts del repo:

```bash
pip install -q "google-cloud-aiplatform>=1.70,<2"

# Entrena AutoML (async, ~2 h server-side). Pasale el ID de tu dataset tabular:
DATASET_ID=<id-del-dataset> python ../scripts/train_automl.py

# Cuando el modelo 'churn-automl' aparezca en Registro de modelos, scorea a todos:
MODEL_ID=<id-del-modelo> python ../scripts/batch_predict_automl.py
```

> **Ojo:** usá estos scripts (SDK), no la opción "AutoML en canalizaciones" de la consola: ese
> template de Google falla por un bug propio. El detalle y los comandos parametrizados están en
> `../gcp/runbook.md`, sección *Clase 4*.

## Cierre

Recorriste el ciclo entero de una punta: **dato → modelo → registro → batch → decisión**. De un CSV
salió una lista priorizada para CRM.

En la **clase 5** damos vuelta la pregunta: ¿cómo hace *otro sistema* para pedirle el score a este
modelo en el momento, sin abrir esta notebook? Ese mismo modelo, detrás de una **API** con contrato
estable.

**Tarea:** corré esta misma notebook con el **dato de tu TFI**. Con que entrenes algo y veas un
score, alcanza. Anotá qué te rompió.